In [1]:
import os

# Prepend the folder containing cdo to PATH
os.environ["PATH"] = "/sw/spack-levante/cdo-2.2.2-4z4icb/bin:" + os.environ["PATH"]

from cdo import Cdo
cdo = Cdo()
print(cdo.version())


2.2.2


In [2]:
!ls /pool/data/ERA5/E5/

E5.bib	ml  pl	sf


In [3]:
import os
import xarray as xr
import pandas as pd
from cdo import Cdo
from joblib import Parallel, delayed

# Paths
data_path_ml = "/pool/data/ERA5/E5/ml/an/1D/"
data_path_pl = "/pool/data/ERA5/E5/pl/an/1D/"
hour_path_sf = "/pool/data/ERA5/E5/sf/an/1H/"
day_path_sf  = "/pool/data/ERA5/E5/sf/an/1D/"

scratch_path = "/scratch/u/u301827/paris/lcl/"
final_path   = "/work/uc1275/u301827/02_MSE/paris/LCL/"

os.makedirs(scratch_path, exist_ok=True)
os.makedirs(final_path,   exist_ok=True)

# Paris coordinates
PARIS_LON = 2.35
PARIS_LAT = 48.86
lon_min = lon_max = PARIS_LON
lat_min = lat_max = PARIS_LAT


In [4]:
def process_era5_paris(year, month, var_num, var,
                       levels="ml", level="137"):
    """
    Process ERA5 daily data selecting ONLY the Paris grid cell.
    
    Notes:
    - ERA5 daily (1D) files in the pool are already aggregated from hourly data,
      so no daily aggregation is necessary here.
    - This function extracts the Paris grid cell and applies vertical level
      selection where required (model or pressure levels).
    """

    cdo = Cdo()
    date_str = f"{year}-{month:02d}"

    # Output folders
    var_scratch = os.path.join(scratch_path, var)
    var_final   = os.path.join(final_path, var)
    os.makedirs(var_scratch, exist_ok=True)
    os.makedirs(var_final,   exist_ok=True)

    out_file = os.path.join(var_final, f"{var}_{date_str}_paris.nc")

    # ---------------------------------------------------------
    # Determine which ERA5 path to use
    # ---------------------------------------------------------
    if levels == "ml":
        data_path = data_path_ml
    elif levels == "pl":
        data_path = data_path_pl
    else:
        data_path = day_path_sf   # surface daily fields

    var_file = f"{data_path}{var_num}/E5{levels}00_1D_{date_str}_{var_num}.grb"
    if not os.path.exists(var_file):
        print(f"Missing: {var_file}")
        return

    # Step 1: Convert GRIB → NetCDF + regular grid
    reg_file = os.path.join(var_scratch, f"reg_{var}_{date_str}.nc")
    cdo.setgridtype("regular", input=var_file, output=reg_file, options="-f nc --eccodes")

    # Step 2: Select only the Paris grid cell
    paris_file = os.path.join(var_scratch, f"paris_{var}_{date_str}.nc")
    # Step 2: select Paris grid cell (nearest grid point)
    cdo.remapnn(f"lon={PARIS_LON}_lat={PARIS_LAT}", input=reg_file, output=paris_file)

    # ---------------------------------------------------------
    # Step 3: Apply vertical level logic
    # ---------------------------------------------------------

    # Model levels (ML)
    if levels == "ml":
        cdo.sellevel(level, input=paris_file, output=out_file)
        os.remove(paris_file)

    # Pressure levels (PL)
    elif levels == "pl":
        ds = xr.open_dataset(paris_file)
        ds_sel = ds.sel(plev=int(level))
        ds_sel["time"] = pd.to_datetime(ds_sel["time"].values)
        ds_sel.to_netcdf(out_file)
        ds.close()
        os.remove(paris_file)

    # Surface fields (SF)
    else:
        # Surface fields (SF)
        import shutil
        shutil.move(paris_file, out_file)

    # Clean up temp file
    os.remove(reg_file)

    return out_file


In [ ]:
# === Variable mapping ===
era5_vars = {
    # Dictionary mapping variable short names to GRIB codes
     133: "q",
     157: "r",
     130: "t",
     129: "z",
}

# === Date range (monthly) ===
start_date = "1940-01-01"
end_date   = "2024-12-31"
months = pd.date_range(start_date, end_date, freq="MS")  # monthly start dates

# === Wrapper function for correct level handling ===
def run_process(date, var_num, var):

    # Pressure-level variables (1000 hPa)
    if var_num in [130, 133, 157, 129]:
        return process_era5_paris(
            year=date.year,
            month=date.month,
            var_num=f"{var_num:03d}",
            var=var,
            levels="pl",
            level="100000"
        )

    else:
        print(f"Variable {var_num} not supported.")
        return None

# === Parallel execution ===
from joblib import Parallel, delayed

n_jobs = 50

results = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(run_process)(month, var_num, var)
    for var_num, var in era5_vars.items()
    for month in months
)

print("Processing completed.")


[Parallel(n_jobs=50)]: Using backend LokyBackend with 50 concurrent workers.
[Parallel(n_jobs=50)]: Done  13 tasks      | elapsed:  1.2min
[Parallel(n_jobs=50)]: Done  28 tasks      | elapsed:  1.3min
[Parallel(n_jobs=50)]: Done  45 tasks      | elapsed:  1.4min
[Parallel(n_jobs=50)]: Done  62 tasks      | elapsed:  2.4min
[Parallel(n_jobs=50)]: Done  81 tasks      | elapsed:  2.5min
[Parallel(n_jobs=50)]: Done 100 tasks      | elapsed:  2.8min
[Parallel(n_jobs=50)]: Done 121 tasks      | elapsed:  3.7min
[Parallel(n_jobs=50)]: Done 142 tasks      | elapsed:  3.9min
[Parallel(n_jobs=50)]: Done 165 tasks      | elapsed:  4.9min
[Parallel(n_jobs=50)]: Done 188 tasks      | elapsed:  5.1min
[Parallel(n_jobs=50)]: Done 213 tasks      | elapsed:  6.2min
[Parallel(n_jobs=50)]: Done 238 tasks      | elapsed:  6.4min
[Parallel(n_jobs=50)]: Done 265 tasks      | elapsed:  7.4min
[Parallel(n_jobs=50)]: Done 292 tasks      | elapsed:  7.7min
[Parallel(n_jobs=50)]: Done 321 tasks      | elapsed:  

In [ ]:
import numpy as np
from scipy.special import lambertw
import xarray as xr

def compute_LCL(T, qv, Rh, z, g=9.81, p0 = 1000):
    """
    Compute the LCL temperature and height from temperature, humidity, and geopotential.
    
    Parameters
    ----------
    T : array-like or xarray.DataArray
        Temperature at reference level (K)
    qv : array-like or xarray.DataArray
        Specific humidity at reference level (kg/kg)
    Rh : array-like or xarray.DataArray
        Relative humidity at reference level (0 to 1)
    z : array-like or xarray.DataArray
        Geopotential at reference level (m^2/s^2), will be divided by g to get height (m)
    g : float, optional
        Gravitational acceleration (m/s²), default is 9.81
    p : float, optional
        pressure level of input (default 1000 hPa

    Returns
    -------
    TLCL : same shape as inputs
        Temperature at LCL (K)
    zLCL : same shape as inputs
        Height of LCL (m)
    """

    # Physical constants
    Ttr = 273.16  # Triple point temperature (K)
    E0v = 2.3740e6  # Latent heat of vaporization at Ttr (J/kg)
    cvl = 4119      # Specific heat liquid water (J/kg/K)
    cvv = 1418      # Specific heat water vapor (J/kg/K)
    Rv  = 461       # Gas constant for water vapor (J/kg/K)
    cpv = cvv + Rv
    Ra  = 287.04    # Gas constant for dry air (J/kg/K)
    cva = 719
    cpa = cva + Ra

    # Moist air specific gas constant and heat capacity
    Rm  = (1 - qv) * Ra + qv * Rv
    cpm = (1 - qv) * cpa + qv * cpv

    # Coefficients for LCL temperature equation
    a = cpm / Rm + (cvl - cpv) / Rv
    b = - (E0v - Ttr * (cvv - cvl)) / (Rv * T)
    c = b / a

    # Argument to Lambert W function
    RHcec = c * np.exp(c) * Rh**(1 / a)

    # Compute TLCL using -1 branch of Lambert W
    TLCL = T * c / lambertw(RHcec, k=-1).real

    # Compute zLCL from TLCL
    z = z / g  # convert geopotential to height
    zLCL = z + (cpm / g) * (T - TLCL)
    zLCL_old = z+cpm/g * (T-55-1/(1/(T-55)-np.log(Rh)/2840))
    
    pLCL =  100.0 * p0 * (TLCL / T) ** (cpm / Rm)

    return TLCL, zLCL, pLCL


## 